In [54]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error


In [55]:
df=pd.read_csv('Ice Cream Sales and Temperature.csv')

In [56]:
df.head()

,Date,Time,Temperature (Celsius),Ice Cream Sales
0,2023-01-01,08:00,18,20
1,2023-01-01,08:05,19,22
2,2023-01-01,08:10,20,25
3,2023-01-01,08:15,21,24
4,2023-01-01,08:20,22,26


In [57]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45 entries, 0 to 44
Data columns (total 4 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Date                   45 non-null     object
 1   Time                   45 non-null     object
 2   Temperature (Celsius)  45 non-null     int64 
 3   Ice Cream Sales        45 non-null     int64 
dtypes: int64(2), object(2)
memory usage: 1.5+ KB


In [58]:
df=df.drop('Date',axis=1)

In [59]:

df["Time"] = pd.to_datetime(df["Time"], format="%H:%M")

df["hour"] = df["Time"].dt.hour
df["minute"] = df["Time"].dt.minute

df.drop("Time", axis=1, inplace=True)

In [60]:
df.columns=['Temperature(C)', 'Ice_Cream_Sales', 'hour', 'minute']

In [61]:
df.head()

,Temperature(C),Ice_Cream_Sales,hour,minute
0,18,20,8,0
1,19,22,8,5
2,20,25,8,10
3,21,24,8,15
4,22,26,8,20


In [62]:
X=df.drop('Ice_Cream_Sales',axis=1)

In [63]:
y=df['Ice_Cream_Sales']

In [64]:
number_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")), 
    ("scaler", StandardScaler())
])

catagorical_pipeline = Pipeline([
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("encoder",OneHotEncoder(handle_unknown="ignore",sparse_output=False))
])

In [65]:
transformer = ColumnTransformer([
                            ("num", number_pipeline, make_column_selector(dtype_include=np.number)),
                            ("cat", catagorical_pipeline, make_column_selector(dtype_include=object))
])

In [66]:
model_pipeline = Pipeline([
                            ("preprocessing", transformer),
    ("poly", PolynomialFeatures(degree=3, include_bias=False)),
                            ("model", LinearRegression())
])

In [67]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=36)

In [68]:
model_pipeline.fit(X_train,y_train)

,steps,"[('preprocessing', ...), ('poly', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [69]:
y_prediction_test = model_pipeline.predict(X_test)
r2= r2_score(y_test, y_prediction_test)
RMSE = np.sqrt(mean_squared_error(y_test, y_prediction_test))

print("R2 Score:", r2)
print("RMSE:", RMSE)

R2 Score: 0.9541993860129206
RMSE: 1.9669108476774113


In [70]:
import pandas as pd

temperature = float(input("Enter Temperature (C): "))
hour = int(input("Enter Hour (0-23): "))
minute = int(input("Enter Minute (0-59): "))

user_data = pd.DataFrame({
    "Temperature(C)": [temperature],
    "hour": [hour],
    "minute": [minute]
})

prediction = model_pipeline.predict(user_data)

print("Predicted Ice Cream Sales:", prediction[0])

Enter Temperature (C):  55
Enter Hour (0-23):  21
Enter Minute (0-59):  32


Predicted Ice Cream Sales: 120.42330915777913


In [71]:
user_data

,Temperature(C),hour,minute
0,55.0,21,32
